# 02 · Collecting a live Instagram sample

**Goal.** Build a sample of real, current Instagram profiles, described by the same 11 features as the Kaggle data, so the trained models can be tested outside the dataset they were built from.

**Sampling**
1. Pick a handful of public posts and reels (`data/seed_posts.txt`), mixing ordinary posts and giveaway/promo posts. They were not chosen by how fake the commenters look.
2. Collect the usernames of **commenters** on those posts with the Apify *Instagram Comment Scraper*.
3. Fetch each commenter's public profile metadata with the Apify *Instagram Profile Scraper*.
4. Turn the metadata into the 11 features, then label each profile by hand (see `LABELING_RUBRIC.md`).

Apify runs on Apify's servers, so the collection doesn't use your own IP or an Instagram login.

The collection cells only run when `RUN_COLLECTION = True` and an `APIFY_TOKEN` is set. By default the notebook rebuilds the feature table from the saved raw data in `data/live_raw.jsonl`, so it runs offline. **Usernames are never printed**; only aggregates are shown.

This notebook is self-contained: all code is defined here.

In [1]:
from pathlib import Path
import os
# Run from the project root (the folder that holds data/ and models/), wherever Jupyter was started.
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)

In [2]:
import hashlib
import json
import re

import numpy as np
import pandas as pd
import requests

RUN_COLLECTION = False   # set True to call Apify again (costs credits, changes the sample)
PER_POST = 80            # max comments collected per seed post

In [3]:
# The 11 features, named as in the Kaggle "Instagram Fake Spammer Genuine Accounts" files.
FEATURES = [
    "profile pic",           # 1 if the account has a (non-default) profile picture
    "nums/length username",  # share of digits in the username
    "fullname words",        # number of words in the full name
    "nums/length fullname",  # share of digits in the full name
    "name==username",        # 1 if full name equals username
    "description length",    # characters in the bio
    "external URL",          # 1 if the profile has an external link
    "private",               # 1 if the account is private
    "#posts",
    "#followers",
    "#follows",
]
LABEL = "fake"  # 1 = fake, 0 = genuine
COUNT_FEATURES = ["description length", "#posts", "#followers", "#follows"]

## 1. Feature extraction
The Kaggle features are recomputed from raw profile metadata. Two need care:
- **Profile picture:** every account has a picture URL, so "no picture" means Instagram's default avatar. It is detected by the avatar's image id in the URL. Instagram changed the default avatar, so both known ids are checked.
- **Digit shares** are rounded to 2 decimals, as in Kaggle.

In [4]:
# Instagram's default "no photo" avatar, identified by its image id in the picture URL.
DEFAULT_AVATAR_MARKERS = [
    "44884218_345707102882519_2446069589734326272_n",   # older default avatar
    "573323465_1219825463302212_7278921664109726296_n",  # default avatar seen Nov 2026
]


def has_pic(url):
    return bool(url) and not any(m in url for m in DEFAULT_AVATAR_MARKERS)


def digit_ratio(s):
    s = s or ""
    return round(sum(c.isdigit() for c in s) / len(s), 2) if s else 0.0


def profile_to_features(username, full_name, biography, external_url, is_private,
                        posts, followers, follows, has_profile_pic):
    """Turn raw public profile metadata into the 11 dataset features."""
    full_name = (full_name or "").strip()
    norm = lambda s: re.sub(r"[^a-z0-9]", "", (s or "").lower())
    return {
        "profile pic": int(bool(has_profile_pic)),
        "nums/length username": digit_ratio(username),
        "fullname words": len(full_name.split()),
        "nums/length fullname": digit_ratio(full_name),
        "name==username": int(bool(full_name) and norm(full_name) == norm(username)),
        "description length": len(biography or ""),
        "external URL": int(bool(external_url)),
        "private": int(bool(is_private)),
        "#posts": int(posts or 0),
        "#followers": int(followers or 0),
        "#follows": int(follows or 0),
    }


def apify_profile_to_features(p):
    return profile_to_features(
        p.get("username"), p.get("fullName"), p.get("biography"), p.get("externalUrl"),
        p.get("private"), p.get("postsCount"), p.get("followersCount"), p.get("followsCount"),
        has_pic(p.get("profilePicUrl") or p.get("profilePicUrlHD")))

## 2. Collection (Apify), guarded by `RUN_COLLECTION`

In [5]:
APIFY = "https://api.apify.com/v2/acts/{}/run-sync-get-dataset-items"


def commenters_of(post_urls, per_post, token):
    """Usernames of people who commented on the given public posts/reels."""
    r = requests.post(APIFY.format("apify~instagram-comment-scraper"), params={"token": token},
                      json={"directUrls": post_urls, "resultsLimit": per_post}, timeout=600)
    r.raise_for_status()
    return [c["ownerUsername"] for c in r.json() if c.get("ownerUsername")]


def profiles_of(usernames, token):
    """Raw public profile metadata, 50 usernames per Apify call."""
    out = []
    for i in range(0, len(usernames), 50):
        r = requests.post(APIFY.format("apify~instagram-profile-scraper"), params={"token": token},
                          json={"usernames": usernames[i:i + 50]}, timeout=600)
        r.raise_for_status()
        out += [p for p in r.json() if p.get("username")]
        print(f"apify: {len(out)} profiles so far")
    return out


if RUN_COLLECTION:
    token = os.environ["APIFY_TOKEN"]
    posts = [l.strip() for l in open("data/seed_posts.txt") if l.strip() and not l.startswith("#")]
    names = list(dict.fromkeys(n.lstrip("@") for n in commenters_of(posts, PER_POST, token)))
    Path("data/live_usernames.csv").write_text("username\n" + "\n".join(names) + "\n")
    raw = profiles_of(names, token)
    with open("data/live_raw.jsonl", "w") as f:
        for p in raw:
            f.write(json.dumps(p, default=str) + "\n")
else:
    print("RUN_COLLECTION is False: using the saved data/live_raw.jsonl")

RUN_COLLECTION is False: using the saved data/live_raw.jsonl


## 3. Rebuild the feature table from the saved raw data

In [6]:
seed_posts = [l.strip() for l in open("data/seed_posts.txt") if l.strip() and not l.startswith("#")]
raw = [json.loads(l) for l in open("data/live_raw.jsonl")]
usernames = pd.read_csv("data/live_usernames.csv")["username"].str.lstrip("@").tolist()
print(f"seed posts: {len(seed_posts)}")
print(f"raw profile records: {len(raw)}")
print(f"usernames in the labeling sample (data/live_usernames.csv): {len(usernames)}")

by_user = {p["username"]: p for p in raw if p.get("username")}
missing = [u for u in usernames if u not in by_user]
print(f"sample usernames with no raw record: {len(missing)}")

seed posts: 20
raw profile records: 284
usernames in the labeling sample (data/live_usernames.csv): 239
sample usernames with no raw record: 0


In [7]:
def short_id(u):
    """Anonymous id, so no usernames appear in the notebook."""
    return hashlib.sha256(u.encode()).hexdigest()[:8]


live = pd.DataFrame([{"id": short_id(u), **apify_profile_to_features(by_user[u])}
                     for u in usernames if u in by_user])
# A record without counts means the account was gone or unavailable when scraped.
live["scrape_empty"] = [int("postsCount" not in by_user[u]) for u in usernames if u in by_user]
print(f"feature table: {len(live)} profiles; empty scrape records: {live.scrape_empty.sum()}")
live.drop(columns="id").describe().T[["mean", "50%", "max"]].round(2)

feature table: 239 profiles; empty scrape records: 1


,mean,50%,max
profile pic,0.93,1.0,1.00
nums/length username,0.11,0.0,0.71
fullname words,1.79,2.0,8.00
nums/length fullname,0.00,0.0,0.36
name==username,0.11,0.0,1.00
description length,41.59,12.0,153.00
external URL,0.14,0.0,1.00
private,0.39,0.0,1.00
#posts,708.05,39.0,50077.00
#followers,33433.48,222.0,5060726.00


### Check: does this match the labeled file?
`data/live_labeled.csv` holds the same profiles with the hand labels added. The rebuilt features should match it exactly.

In [8]:
labeled = pd.read_csv("data/live_labeled.csv")
assert labeled["username"].tolist() == [u for u in usernames if u in by_user]
mismatch = {c: int((labeled[c].values != live[c].values).sum()) for c in FEATURES}
print("feature mismatches vs live_labeled.csv:", mismatch)
print(f"labeled: {labeled[LABEL].notna().sum()} of {len(labeled)} "
      f"({int(labeled[LABEL].sum())} fake, {labeled[LABEL].mean():.1%} of labeled)")

feature mismatches vs live_labeled.csv: {'profile pic': 0, 'nums/length username': 0, 'fullname words': 0, 'nums/length fullname': 0, 'name==username': 0, 'description length': 0, 'external URL': 0, 'private': 0, '#posts': 0, '#followers': 0, '#follows': 0}
labeled: 221 of 239 (26 fake, 11.8% of labeled)


## 4. Sample summary

In [9]:
def summary(df):
    return pd.Series({
        "profiles": len(df),
        "% private": 100 * df["private"].mean(),
        "% no profile pic": 100 * (1 - df["profile pic"].mean()),
        "% external URL": 100 * df["external URL"].mean(),
        "% 0 posts": 100 * (df["#posts"] == 0).mean(),
        "% username has digits": 100 * (df["nums/length username"] > 0).mean(),
        "median #posts": df["#posts"].median(),
        "median #followers": df["#followers"].median(),
        "median #follows": df["#follows"].median(),
    }).round(1)

lab = labeled.copy()
pd.DataFrame({
    "whole sample": summary(live),
    "labeled genuine": summary(lab[lab[LABEL] == 0]),
    "labeled fake": summary(lab[lab[LABEL] == 1]),
    "unlabeled": summary(lab[lab[LABEL].isna()]),
})

,whole sample,labeled genuine,labeled fake,unlabeled
profiles,239.0,195.0,26.0,18.0
% private,39.3,37.4,34.6,66.7
% no profile pic,6.7,2.6,23.1,27.8
% external URL,14.2,17.4,0.0,0.0
% 0 posts,17.6,11.3,50.0,38.9
% username has digits,41.0,37.4,69.2,38.9
median #posts,39.0,58.0,0.5,13.5
median #followers,222.0,270.0,65.5,53.0
median #follows,603.0,603.0,982.0,404.0


**Notes**
- About 37% of the profiles are private. For those the labeler sees only the profile header, which makes labeling harder.
- The Apify profile scraper also returns `externalUrls` (a list of link objects). The feature uses `externalUrl` only, so a few profiles with links only in the list count as having no external URL.
- Scraped metadata and usernames are personal data: `data/live_raw.jsonl` and the username files are kept private and are not committed.
- Labeling follows `LABELING_RUBRIC.md`. Notebook 03 evaluates the models on the labeled sample.